# Retrocausality Mini Project

**当前规模**：3 agents, 5×5 grid, 7 steps, Mesa 2.3.0, PyTorch TCN-based

目标：  
Agent 0 使用 TCN 预测未来碰撞 → 实时调整当前移动，避免碰撞；  
其他两个 agent 随机移动（allow_collisions=True 时倾向碰撞）。

主要功能：
- 生成混合数据（易碰撞 + 难碰撞）
- 训练 / 加载 TCN
- 对比：纯随机 vs 有 TCN 规则 的碰撞次数

In [1]:
# ==============================================
# 导入（只保留当前 mini 项目真正需要的）
# ==============================================
import random
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pickle
from tqdm import tqdm

# Mesa（已固定版本 2.3.0）
from mesa import Agent, Model
from mesa.space import MultiGrid
from mesa.time import RandomActivation

# ====================== 项目配置与模块 ======================
# 先设置工作目录（放在最前面，避免路径问题）
# 在 Jupyter notebook 中获取项目根目录
# notebook 在 notebooks/ 子目录中，需要上升一级
import os
import sys
from pathlib import Path

current_dir = Path.cwd()
project_root = current_dir.parent if current_dir.name == 'notebooks' else current_dir
os.chdir(project_root)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"工作目录已设置为: {os.getcwd()}")

# ================== 导入 config（必须在最前面） ==================
from config import *

# ================== 导入项目核心模块（已修复） ==================
from src.model import RetroAgent, RetroModel, TCN
from src.tcn import train, prepare_training_data          # ← 关键修改
# from src.data_gen import collect_mixed_data             # 如果你还没实现，先注释掉
# from src.evaluate import compare_rule_effects           # 如果还没实现，先注释掉
from src.visualize import create_animation

# ====================== 固定随机种子 ======================
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.backends.cudnn.deterministic = True

print("✅ 导入完成！")
print(f"   GRID: {GRID_WIDTH}x{GRID_HEIGHT}, Agents: {NUM_AGENTS}, Steps: {STEPS}")
print(f"   TCN Model Path: {TCN_MODEL_PATH}")
print(f"   Data Path: {MIXED_DATA_PATH}")

工作目录已设置为: e:\projects\Dynamic-Retrocausal-Simulator
BASE_DIR resolved to: E:\projects\Dynamic-Retrocausal-Simulator
DATA_DIR       : E:\projects\Dynamic-Retrocausal-Simulator\data
MIXED_DATA_PATH: E:\projects\Dynamic-Retrocausal-Simulator\data\abm_mixed_3agents_5x5_collisions.pkl
TCN_MODEL_PATH : E:\projects\Dynamic-Retrocausal-Simulator\models\tcn_best.pt
Current working dir: e:\projects\Dynamic-Retrocausal-Simulator
✅ 导入完成！
   GRID: 5x5, Agents: 3, Steps: 7
   TCN Model Path: E:\projects\Dynamic-Retrocausal-Simulator\models\tcn_best.pt
   Data Path: E:\projects\Dynamic-Retrocausal-Simulator\data\abm_mixed_3agents_5x5_collisions.pkl


In [ ]:
import mesa
import torch
import pandas as pd
import tqdm
print(mesa.__version__)          # 2.3.0
print(torch.__version__)         # 2.7.0
print(pd.__version__)            # 3.0.1
print(tqdm.__version__)          # 4.67.3
print("STEPS:", STEPS)                # 7
print("SEQ_LEN:", SEQ_LEN)            # 5
print("device:", device)              # cuda

## 1. 生成小规模测试数据（调试用）

In [ ]:
# ==============================================
# 2. 生成小规模训练数据（调试用）
# ==============================================
from src.data_gen import generate_data
import pathlib

print("🚀 开始生成小批量混合训练数据...")

# 小批量参数（快速测试用，正式训练再调大）
small_data = generate_data(
    num_runs=1200,           # 先用 800，够用了
    steps_per_run=STEPS,
    filename=MIXED_DATA_PATH,
    collision_boost=True    # 关键参数：开启碰撞增强
)

# 简单统计碰撞情况
total_collisions = 0
total_steps = 0

for run in small_data:
    for record in run:
        if record.get('collision', 0) == 1:
            total_collisions += 1
        total_steps += 1

collision_rate = total_collisions / total_steps if total_steps > 0 else 0

print(f"✅ 小批量数据生成完成！")
print(f"   生成 runs: {len(small_data)}")
print(f"   总数据点: {total_steps}")
print(f"   总碰撞次数: {total_collisions}")
print(f"   碰撞率: {collision_rate:.4f} ({collision_rate*100:.2f}%)")
print(f"   数据已保存到: {MIXED_DATA_PATH}")

# 可选：查看前几个数据样本
print("\n前3条数据示例（Agent 0）:")
agent0_samples = [d for d in small_data[0] if d['agent_id'] == 0][:3]
for sample in agent0_samples:
    print(sample)

🚀 开始生成小批量混合训练数据...
🚀 开始生成 800 runs 数据...


Generating: 100%|██████████| 800/800 [00:00<00:00, 15167.97it/s]


✅ 数据生成完成！
   Total runs: 800
   Total data points: 0
   Total collisions (Agent 0): 0
   Collision rate: 0.0000 (0.00%)
✅ 小批量数据生成完成！
   生成 runs: 800
   总数据点: 0
   总碰撞次数: 0
   碰撞率: 0.0000 (0.00%)
   数据已保存到: E:\projects\Dynamic-Retrocausal-Simulator\data\abm_mixed_3agents_5x5_collisions.pkl

前3条数据示例（Agent 0）:


In [ ]:
# ==============================================
# 3. 训练 TCN 模型（核心步骤）
# ==============================================
from src.tcn import train
train()

## 2. 检查单次模拟轨迹（调试用）

In [ ]:
# 手动跑一次模拟，看 Agent 0 是否使用了 TCN 规则
model = RetroModel(allow_collisions=True, tcn_path="models/tcn_best.pth")
data, coll_count = model.run_simulation(steps=7)

print(f"本次模拟 Agent 0 碰撞次数：{coll_count}")

# 打印 Agent 0 的移动序列
agent0_moves = [d['pos'] for d in data if d['agent_id'] == 0]
print("Agent 0 位置序列：", agent0_moves)

## 3. 对比实验（核心验证 retrocausality）

In [ ]:
# 小规模对比（快速看效果）
no_rule_coll, with_rule_coll = compare_rule_effects(
    num_sims=50,
    steps=STEPS,
    tcn_path="models/tcn_best.pth"   # 改成你最新的模型路径
)

reduction = (no_rule_coll - with_rule_coll) / no_rule_coll * 100 if no_rule_coll > 0 else 0
print(f"\n碰撞减少比例：{reduction:.1f}%")

# 如果效果明显，可以再跑大批量
# compare_rule_effects(num_sims=200, steps=STEPS, tcn_path=...) 

## 4. 快速检查 TCN 预测（调试模型）

In [ ]:
# 加载模型并测试单条输入
tcn = load_tcn("models/tcn_best.pth")

# 假设你有某条序列数据（形状 [1, SEQ_LEN, features]）
# 这里用随机假数据演示，实际请替换成你的真实输入
dummy_input = torch.randn(1, SEQ_LEN, 18).to(device)  # 改成你的实际特征维度

with torch.no_grad():
    pred = tcn(dummy_input)
    print("TCN 示例输出：", pred)
    
    # 如果是二分类碰撞预测
    prob = torch.sigmoid(pred).item()
    print(f"预测碰撞概率：{prob:.3f}")